#### 크롤링 예제
1. requests 라이브러리를 이용해서 `moons-86.iptime.org:8080` 요청을 보낸다
2. 응답 받은 데이터를 BeautifulSoup를 이용하여 데이터를 파싱
3. id가 product_1001인 태그를 찾아서 h2, p의 모든 콘텐츠 데이터를 추출한다
3. 모든 상품의 정보를 추출
    - 데이터프레임으로 생성
    - csv 파일로 저장
    

In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd

In [2]:
# 1. 서버에게 요청을 보낸다
res = requests.get('http://moons-86.iptime.org:8080/')


ConnectTimeout: HTTPConnectionPool(host='moons-86.iptime.org', port=8080): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPConnection(host='moons-86.iptime.org', port=8080) at 0x27beb905390>, 'Connection to moons-86.iptime.org timed out. (connect timeout=None)'))

In [ ]:
res

<Response [200]>

In [ ]:
res.text

'<!DOCTYPE html>\n<html lang="ko">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>웹 크롤링 수업용 샘플 페이지</title>\n    <style>\n        body { font-family: \'Malgun Gothic\', sans-serif; line-height: 1.6; padding: 20px; background-color: #f4f4f4; color: #333; }\n        .container { max-width: 960px; margin: 0 auto; background-color: #fff; padding: 30px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }\n        header { text-align: center; margin-bottom: 30px; }\n        h1#page-title { color: #2c3e50; font-size: 2.5em; margin-bottom: 10px; }\n        p#page-description { color: #7f8c8d; font-size: 1.1em; }\n        .product-list { display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 25px; margin-top: 30px; }\n        .product-item { border: 1px solid #e0e0e0; border-radius: 8px; padding: 20px; background-color: #fff; transition: transform 0.2s ease-in-out; }\n        .product-

In [ ]:
#2. BeautifulSoup을 이용해서 데이터를 파싱
soup = bs(res.text, 'html.parser')

In [ ]:
soup

<!DOCTYPE html>

<html lang="ko">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<title>웹 크롤링 수업용 샘플 페이지</title>
<style>
        body { font-family: 'Malgun Gothic', sans-serif; line-height: 1.6; padding: 20px; background-color: #f4f4f4; color: #333; }
        .container { max-width: 960px; margin: 0 auto; background-color: #fff; padding: 30px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
        header { text-align: center; margin-bottom: 30px; }
        h1#page-title { color: #2c3e50; font-size: 2.5em; margin-bottom: 10px; }
        p#page-description { color: #7f8c8d; font-size: 1.1em; }
        .product-list { display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 25px; margin-top: 30px; }
        .product-item { border: 1px solid #e0e0e0; border-radius: 8px; padding: 20px; background-color: #fff; transition: transform 0.2s ease-in-out; }
        .product-item:hover { transform: transl

In [ ]:
# id가 product-1001인 태그를 찾아라 -> div태그에서 id가 product-1001
div_data = soup.find(
    'div',
    attrs = {
        'id' : 'product-1001'
    }
)


In [ ]:
div_data

<div class="product-item" id="product-1001">
<h2 class="product-title">스마트 워치 X100</h2>
<p class="product-category">카테고리: 전자기기</p>
<p class="product-price">가격: 150,000원</p>
<p class="product-rating">평점: ★★★★☆ (4.5/5)</p>
<a class="product-link" href="/products/1001">상세 보기</a>
</div>

In [ ]:
div_data.get_text().split('\n')


['',
 '스마트 워치 X100',
 '카테고리: 전자기기',
 '가격: 150,000원',
 '평점: ★★★★☆ (4.5/5)',
 '상세 보기',
 '']

In [ ]:
# h2, p 태그를 모두 찾아서 각각 텍스트 추출
p_list = div_data.find_all(
    ['h2','p']
)

In [ ]:
p_list

[<h2 class="product-title">스마트 워치 X100</h2>,
 <p class="product-category">카테고리: 전자기기</p>,
 <p class="product-price">가격: 150,000원</p>,
 <p class="product-rating">평점: ★★★★☆ (4.5/5)</p>]

In [ ]:
# p_list에서 각각의 원소들의 콘텐츠 데이터를 추출
[p.get_text() for p in p_list]

['스마트 워치 X100', '카테고리: 전자기기', '가격: 150,000원', '평점: ★★★★☆ (4.5/5)']

In [ ]:
list(
    map(
        lambda x : x.get_text(),
        p_list
    )
)

['스마트 워치 X100', '카테고리: 전자기기', '가격: 150,000원', '평점: ★★★★☆ (4.5/5)']

In [ ]:
# 사이트의 모든 상품의 정보를 가져온다.
# 접근 방법 1 -> div 태그 중 class가 product-item인 태그를 모두 찾는다
div_list = soup.find_all(
    'div',
    attrs = {
        'class' : 'product-item'
    }
)

In [ ]:
# div_list는 각각의 원소가 아이템의 정보를 가지고 있다.
# div_list[0] # id가 product-1001과 같은 태그 -> 위의 작업과 동일한 작업을 반복 실행하여 빈 리스트에 결과를 추가

values = []

for div in div_list:
    # h2, p 태그를 모두 찾는다
    p_list = div.find_all(
        ['h2','p']
    )
    # p_list에서 각각의 원소들의 텍스트 추출
    value = [p.get_text() for p in p_list]
    # values에 value를 추가
    values.append(value)
values

[['스마트 워치 X100', '카테고리: 전자기기', '가격: 150,000원', '평점: ★★★★☆ (4.5/5)'],
 ['무선 이어폰 Pro', '카테고리: 전자기기', '가격: 89,000원', '평점: ★★★★★ (5.0/5)'],
 ['고급 가죽 지갑', '카테고리: 패션 잡화', '가격: 75,000원', '평점: ★★★★☆ (4.0/5)'],
 ['에코백 디자인 컬렉션', '카테고리: 패션 잡화', '가격: 25,000원', '평점: ★★★★☆ (4.2/5)'],
 ['유기농 커피 원두 500g', '카테고리: 식품', '가격: 18,000원', '평점: ★★★★★ (4.8/5)']]

In [ ]:
df = pd.DataFrame(
    values,columns = ['상품명','카테고리','가격','평점']
)

In [ ]:
df

,상품명,카테고리,가격,평점
0,스마트 워치 X100,카테고리: 전자기기,"가격: 150,000원",평점: ★★★★☆ (4.5/5)
1,무선 이어폰 Pro,카테고리: 전자기기,"가격: 89,000원",평점: ★★★★★ (5.0/5)
2,고급 가죽 지갑,카테고리: 패션 잡화,"가격: 75,000원",평점: ★★★★☆ (4.0/5)
3,에코백 디자인 컬렉션,카테고리: 패션 잡화,"가격: 25,000원",평점: ★★★★☆ (4.2/5)
4,유기농 커피 원두 500g,카테고리: 식품,"가격: 18,000원",평점: ★★★★★ (4.8/5)


In [ ]:
# Series에서 replace()함수의 기준은? -> 각각의 value
# 문자를 기준으로 replace를 사용하려면? 1. 반복문을 이용 2. map() 함수를 이용 3. .str을 이용하여 문자열 함수에 접근
for i in df['카테고리']:
    print(i.replace('카테고리: ',''))


전자기기
전자기기
패션 잡화
패션 잡화
식품


In [ ]:
# python에 내장된 map()함수는 데이터를 가지고 있지 않기 때문에 인자에 데이터를 입력
# Series에 내장된 map() 함수는 Series안에 데이터가 이미 존재하기 때문에 인자에는 함수만 입력
df['가격'].map(
    lambda x : x.replace('가격: ','')
)

0    150,000원
1     89,000원
2     75,000원
3     25,000원
4     18,000원
Name: 가격, dtype: object

In [ ]:
df['평점'].str.replace('평점: ','')

0    ★★★★☆ (4.5/5)
1    ★★★★★ (5.0/5)
2    ★★★★☆ (4.0/5)
3    ★★★★☆ (4.2/5)
4    ★★★★★ (4.8/5)
Name: 평점, dtype: object

In [ ]:
df

,상품명,카테고리,가격,평점
0,스마트 워치 X100,카테고리: 전자기기,"가격: 150,000원",평점: ★★★★☆ (4.5/5)
1,무선 이어폰 Pro,카테고리: 전자기기,"가격: 89,000원",평점: ★★★★★ (5.0/5)
2,고급 가죽 지갑,카테고리: 패션 잡화,"가격: 75,000원",평점: ★★★★☆ (4.0/5)
3,에코백 디자인 컬렉션,카테고리: 패션 잡화,"가격: 25,000원",평점: ★★★★☆ (4.2/5)
4,유기농 커피 원두 500g,카테고리: 식품,"가격: 18,000원",평점: ★★★★★ (4.8/5)


In [ ]:
# 데이터프레임에서 map()함수를 이용하면 value들이 각각 입력
df.map(
    lambda x : print(x.split(':')[-1].lstrip())
)

스마트 워치 X100
무선 이어폰 Pro
고급 가죽 지갑
에코백 디자인 컬렉션
유기농 커피 원두 500g
전자기기
전자기기
패션 잡화
패션 잡화
식품
150,000원
89,000원
75,000원
25,000원
18,000원
★★★★☆ (4.5/5)
★★★★★ (5.0/5)
★★★★☆ (4.0/5)
★★★★☆ (4.2/5)
★★★★★ (4.8/5)


,상품명,카테고리,가격,평점
0,None,None,None,None
1,None,None,None,None
2,None,None,None,None
3,None,None,None,None
4,None,None,None,None


In [ ]:
# 데이터프레임에서 map()함수를 이용하면 value들이 각각 입력
df.map(
    lambda x : x.split(':')[-1].lstrip()
)

,상품명,카테고리,가격,평점
0,스마트 워치 X100,전자기기,"150,000원",★★★★☆ (4.5/5)
1,무선 이어폰 Pro,전자기기,"89,000원",★★★★★ (5.0/5)
2,고급 가죽 지갑,패션 잡화,"75,000원",★★★★☆ (4.0/5)
3,에코백 디자인 컬렉션,패션 잡화,"25,000원",★★★★☆ (4.2/5)
4,유기농 커피 원두 500g,식품,"18,000원",★★★★★ (4.8/5)


In [ ]:
# 접근 방식2 -> 상품명, 카테고리, 가격, 평점, 하이퍼링크를 각각의 class의 값들로 접근 하여 데이터프레임 생성

# product-list class를 가진 div를 추출( 모든 상품의 정보가 담겨있는 div 영역을 선택 )

div_data = soup.find(
    'div',
    attrs = {
        'class':'product-list'
    }
)

In [ ]:
div_data

<div class="product-list">
<!-- 상품 1 -->
<div class="product-item" id="product-1001">
<h2 class="product-title">스마트 워치 X100</h2>
<p class="product-category">카테고리: 전자기기</p>
<p class="product-price">가격: 150,000원</p>
<p class="product-rating">평점: ★★★★☆ (4.5/5)</p>
<a class="product-link" href="/products/1001">상세 보기</a>
</div>
<!-- 상품 2 -->
<div class="product-item" id="product-1002">
<h2 class="product-title">무선 이어폰 Pro</h2>
<p class="product-category">카테고리: 전자기기</p>
<p class="product-price">가격: 89,000원</p>
<p class="product-rating">평점: ★★★★★ (5.0/5)</p>
<a class="product-link" href="/products/1002">상세 보기</a>
</div>
<!-- 상품 3 -->
<div class="product-item" id="product-1003">
<h2 class="product-title">고급 가죽 지갑</h2>
<p class="product-category">카테고리: 패션 잡화</p>
<p class="product-price">가격: 75,000원</p>
<p class="product-rating">평점: ★★★★☆ (4.0/5)</p>
<a class="product-link" href="/products/1003">상세 보기</a>
</div>
<!-- 상품 4 -->
<div class="product-item" id="product-1004">
<h2 class="product-title"

In [ ]:
# 상품명에 접근 : class의 값이 product-title인 태그에 접근(모드 찾는다.)
title_list = div_data.find_all(
    attrs = {
        'class' : 'product-title'
    }
)

In [ ]:
title_list

[<h2 class="product-title">스마트 워치 X100</h2>,
 <h2 class="product-title">무선 이어폰 Pro</h2>,
 <h2 class="product-title">고급 가죽 지갑</h2>,
 <h2 class="product-title">에코백 디자인 컬렉션</h2>,
 <h2 class="product-title">유기농 커피 원두 500g</h2>]

In [ ]:
[title.get_text() for title in title_list]

['스마트 워치 X100', '무선 이어폰 Pro', '고급 가죽 지갑', '에코백 디자인 컬렉션', '유기농 커피 원두 500g']

In [ ]:
# a 태그에서 href라는 속성의 값을 추출하려면? -> Tag['속성명']
link_list = div_data.find_all(
    attrs = {
        'class' : 'product-link'
    }
)

In [ ]:
link_list

[<a class="product-link" href="/products/1001">상세 보기</a>,
 <a class="product-link" href="/products/1002">상세 보기</a>,
 <a class="product-link" href="/products/1003">상세 보기</a>,
 <a class="product-link" href="/products/1004">상세 보기</a>,
 <a class="product-link" href="/products/1005">상세 보기</a>]

In [ ]:
links = []
for link in link_list:
    # print('https://moons-86.iptime.org:8080' + link['href'])
    links.append('https://moons-86.iptime.org:8080' + link['href'])
links

['https://moons-86.iptime.org:8080/products/1001',
 'https://moons-86.iptime.org:8080/products/1002',
 'https://moons-86.iptime.org:8080/products/1003',
 'https://moons-86.iptime.org:8080/products/1004',
 'https://moons-86.iptime.org:8080/products/1005']

In [ ]:
class_list = ['product-title','product-category','product-price','product-rating','product-link']
base_url = 'https://moons-86.iptime.org:8080'
dict_data = {}

# 반복 실행하면서 빈 딕셔너리에 key : [] 값들을 추가
for cls in class_list:
    _list = div_data.find_all(
        attrs = {
            'class' : cls
        }
    )
    # cls가 product-link라면 href 속성의 값들을 리스트로 생성
    if cls == 'product-link':
        value = [base_url + link['href'] for link in _list]
    else:
        value = [ data.get_text().split(':')[-1].strip() for data in _list ]

    # key 값은 cls에서 -로 잘라주고 뒤의 값들을 사용
    key_data = cls.split('-')[-1]

    dict_data[key_data] = value
dict_data

{'title': ['스마트 워치 X100',
  '무선 이어폰 Pro',
  '고급 가죽 지갑',
  '에코백 디자인 컬렉션',
  '유기농 커피 원두 500g'],
 'category': ['전자기기', '전자기기', '패션 잡화', '패션 잡화', '식품'],
 'price': ['150,000원', '89,000원', '75,000원', '25,000원', '18,000원'],
 'rating': ['★★★★☆ (4.5/5)',
  '★★★★★ (5.0/5)',
  '★★★★☆ (4.0/5)',
  '★★★★☆ (4.2/5)',
  '★★★★★ (4.8/5)'],
 'link': ['https://moons-86.iptime.org:8080/products/1001',
  'https://moons-86.iptime.org:8080/products/1002',
  'https://moons-86.iptime.org:8080/products/1003',
  'https://moons-86.iptime.org:8080/products/1004',
  'https://moons-86.iptime.org:8080/products/1005']}

In [ ]:
df = pd.DataFrame(dict_data)

In [ ]:
# csv 파일로 저장시 인덱스가 포함되어 저장 -> 의미가 없는 인덱스 일수도 있지만 의미가 존재하는 경우
# index 매개변수 존재 -> 인덱스를 저장할것인가?
df.to_csv('test.csv',index = False)

In [ ]:
# csv 파일에 인덱스(이름 없는 인덱스) 부분이 저장되어있는 경우 인덱스 부분도 컬럼으로 인식해서 로드
# index_col 매개변수 : 인덱스로 사용할 컬럼을 선택(위치/ 컬럼명) 다중 선택 가능
# usecols 매개변수 : 데이터프레임에서 컬럼의 필터 (위치/컬럼명)
pd.read_csv('test.csv', usecols=['title','category'])

,title,category
0,스마트 워치 X100,전자기기
1,무선 이어폰 Pro,전자기기
2,고급 가죽 지갑,패션 잡화
3,에코백 디자인 컬렉션,패션 잡화
4,유기농 커피 원두 500g,식품
